In [1]:

from pathlib import Path

ROOT = Path("/home/ubuntu/avito")
BASELINE = ROOT / "official_baseline/catboost_baseline.py"

EXPERIMENTS = ROOT / "experiments"
EXPERIMENTS.mkdir(exist_ok=True)

source = BASELINE.read_text()

changes = {
    '"task_type": "GPU"': '"task_type": "CPU"',
    '"gpu_ram_part": 0.8,': '',
    '"devices": "0",': '',
    '"num_trees": 2000': '"num_trees": 150',
    '"learning_rate": 0.4': '"learning_rate": 0.1',
    'DATA_PARAMS = {"train_feeds": 0, "val_feeds": 0}':
    'DATA_PARAMS = {"train_feeds": 1000, "val_feeds": 200}',
}

for old, new in changes.items():
    assert old in source, f"Не найден параметр: {old}"
    source = source.replace(old, new, 1)

script = EXPERIMENTS / "catboost_smoke.py"
script.write_text(source)

print("Создан:", script)
print("Официальный baseline не изменён.")


Создан: /home/ubuntu/avito/experiments/catboost_smoke.py
Официальный baseline не изменён.


In [2]:

import subprocess
import sys

output = ROOT / "experiments/runs/B0_smoke_cpu"

assert not output.exists(), (
    f"{output} уже существует. "
    "Выбери новое имя запуска."
)

subprocess.run(
    [
        sys.executable,
        str(script),
        "train",
        "--data", str(ROOT / "competition"),
        "--output", str(output),
    ],
    check=True,
)

print("Обучение завершено!")


Groupwise loss function. OneHotMaxSize set to 10
0:	test: 0.1812507	best: 0.1812507 (0)	total: 94.5ms	remaining: 14.1s
100:	test: 0.2885327	best: 0.2915805 (94)	total: 4.06s	remaining: 1.97s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.2915805339
bestIteration = 94

Shrink model to first 95 iterations.
Обучение завершено!


2026-11-03 21:15:50.544 | INFO     | __main__:train:162 - {'val_ndcg': 0.3500022598054208, 'val_ndcg_at_30': 0.2915805339251254, 'val_ndcg_at_10': 0.21572821764432912, 'train_feeds': 1000, 'train_candidates': 76004, 'val_feeds': 200, 'val_candidates': 16902, 'trees': 95}


# B1 решение 


In [1]:

from pathlib import Path
import importlib.util

ROOT = Path("/home/ubuntu/avito")
script = ROOT / "experiments/catboost_h1_smoke.py"

spec = importlib.util.spec_from_file_location("h1", script)
h1 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(h1)

preview = h1.candidate_dataset(
    ROOT / "competition/train",
    max_feeds=1
).collect(streaming=True)

print(
    preview.select(
        "item_id",
        "category_id",
        "hist_cat_count",
        "target"
    ).head(10)
)

print("Shape:", preview.shape)
print("Количество фичей:", len(h1.FEATURES))

assert preview.height == 72
assert preview["hist_cat_count"][0] == 60
assert preview["hist_cat_count"][1] == 68

print("H1: все проверки пройдены!")


shape: (10, 4)
┌───────────┬─────────────┬────────────────┬────────┐
│ item_id   ┆ category_id ┆ hist_cat_count ┆ target │
│ ---       ┆ ---         ┆ ---            ┆ ---    │
│ i32       ┆ i8          ┆ i64            ┆ i8     │
╞═══════════╪═════════════╪════════════════╪════════╡
│ 73328213  ┆ 3           ┆ 60             ┆ 0      │
│ 47564870  ┆ 13          ┆ 68             ┆ 0      │
│ 92744465  ┆ 11          ┆ 40             ┆ 0      │
│ 69672495  ┆ 7           ┆ 7              ┆ 1      │
│ 92017650  ┆ 13          ┆ 68             ┆ 0      │
│ 106518740 ┆ 13          ┆ 68             ┆ 0      │
│ 8090156   ┆ 13          ┆ 68             ┆ 0      │
│ 51004721  ┆ 13          ┆ 68             ┆ 0      │
│ 101490799 ┆ 13          ┆ 68             ┆ 0      │
│ 66553648  ┆ 11          ┆ 40             ┆ 0      │
└───────────┴─────────────┴────────────────┴────────┘
Shape: (72, 13)
Количество фичей: 10
H1: все проверки пройдены!


In [2]:

from pathlib import Path
import subprocess
import sys

ROOT = Path("/home/ubuntu/avito")

script_h1 = ROOT / "experiments/catboost_h1_smoke.py"
output_h1 = ROOT / "experiments/runs/H1_smoke_cpu"

assert script_h1.exists(), "Скрипт H1 не найден"
assert not output_h1.exists(), "H1 уже запускался: выбери новый output"

subprocess.run(
    [
        sys.executable,
        str(script_h1),
        "train",
        "--data", str(ROOT / "competition"),
        "--output", str(output_h1),
    ],
    check=True,
)

print("H1 обучен!")
print("Модель:", output_h1 / "model.cbm")


Groupwise loss function. OneHotMaxSize set to 10
0:	test: 0.1869998	best: 0.1869998 (0)	total: 89.5ms	remaining: 13.3s
100:	test: 0.2852658	best: 0.2930684 (59)	total: 4.08s	remaining: 1.98s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.2930684157
bestIteration = 59

Shrink model to first 60 iterations.
H1 обучен!
Модель: /home/ubuntu/avito/experiments/runs/H1_smoke_cpu/model.cbm


2026-11-03 22:28:09.817 | INFO     | __main__:train:199 - {'val_ndcg': 0.3459577793345863, 'val_ndcg_at_30': 0.29306841574265957, 'val_ndcg_at_10': 0.21496604465739302, 'train_feeds': 1000, 'train_candidates': 76004, 'val_feeds': 200, 'val_candidates': 16902, 'trees': 60}


In [3]:

import json
from pathlib import Path

root = Path("/home/ubuntu/avito/experiments/runs")

b0 = json.loads((root / "B0_smoke_cpu/run.json").read_text())
h1 = json.loads((root / "H1_smoke_cpu/run.json").read_text())

for key in ["random_seed", "learning_rate", "depth"]:
    print(
        key,
        "B0:", b0["model_params"].get(key),
        "H1:", h1["model_params"].get(key)
    )

print("B0 features:", b0["features"])
print("H1 features:", h1["features"])


random_seed B0: 666 H1: 666
learning_rate B0: 0.10000000149011612 H1: 0.10000000149011612
depth B0: 6 H1: 6
B0 features: ['f1', 'f2', 'f3', 'f4', 'f5', 'embedding_dot', 'embedding_cosine', 'embedding_missing', 'category_id']
H1 features: ['f1', 'f2', 'f3', 'f4', 'f5', 'embedding_dot', 'embedding_cosine', 'embedding_missing', 'hist_cat_count', 'category_id']


In [4]:

import numpy as np
import polars as pl
from catboost import CatBoostRanker, Pool
import importlib.util

ROOT = Path("/home/ubuntu/avito")

def load_script(name, filename):
    spec = importlib.util.spec_from_file_location(name, filename)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

base = load_script("base", ROOT / "experiments/catboost_smoke.py")
hist = load_script("hist", ROOT / "experiments/catboost_h1_smoke.py")

def get_predictions(module, run_name):
    data = module.candidate_dataset(
        ROOT / "competition/val", max_feeds=200
    ).collect(streaming=True)

    model = CatBoostRanker()
    model.load_model(str(
        ROOT / "experiments/runs" / run_name / "model.cbm"
    ))

    pool = Pool(
        data.select(module.FEATURES),
        cat_features=module.CAT_FEATURES
    )

    return data.with_columns(
        pl.Series("score", model.predict(pool))
    )

pred_b0 = get_predictions(base, "B0_smoke_cpu")
pred_h1 = get_predictions(hist, "H1_smoke_cpu")

assert pred_b0["feed_id"].to_list() == pred_h1["feed_id"].to_list()
assert pred_b0["item_id"].to_list() == pred_h1["item_id"].to_list()

def feed_ndcg(y, scores):
    y = np.asarray(y)
    scores = np.asarray(scores)

    order = np.lexsort((y, -scores))
    ranks = np.arange(1, len(y) + 1)

    dcg = (y[order] / np.log2(ranks + 1)).sum()

    positives = int(y.sum())
    if positives == 0:
        return np.nan

    ideal_ranks = np.arange(1, min(30, positives) + 1)
    idcg = (1 / np.log2(ideal_ranks + 1)).sum()

    return dcg / idcg

def per_feed_scores(df):
    return {
        feed_id: feed_ndcg(
            group["target"].to_numpy(),
            group["score"].to_numpy()
        )
        for feed_id, group in df.group_by("feed_id")
    }

scores_b0 = per_feed_scores(pred_b0)
scores_h1 = per_feed_scores(pred_h1)

feed_ids = sorted(scores_b0)

a = np.array([scores_b0[f] for f in feed_ids])
b = np.array([scores_h1[f] for f in feed_ids])

print("B0:", a.mean())
print("H1:", b.mean())
print("Delta:", (b - a).mean())
print("Лент улучшилось:", int((b > a).sum()))
print("Лент ухудшилось:", int((b < a).sum()))


B0: 0.3500022598054208
H1: 0.34595777933458666
Delta: -0.004044480470834184
Лент улучшилось: 90
Лент ухудшилось: 62


In [5]:

import pandas as pd
import numpy as np

diagnostics = pd.DataFrame({
    "feed_id": feed_ids,
    "B0": a,
    "H1": b,
})

diagnostics["delta"] = diagnostics["H1"] - diagnostics["B0"]

print("Худшие 10 лент для H1:")
display(diagnostics.nsmallest(10, "delta"))

print("Лучшие 10 лент для H1:")
display(diagnostics.nlargest(10, "delta"))

positive = diagnostics.loc[diagnostics.delta > 0, "delta"]
negative = diagnostics.loc[diagnostics.delta < 0, "delta"]

print("Средний выигрыш:", positive.mean())
print("Средний проигрыш:", negative.mean())
print("Медианное изменение:", diagnostics.delta.median())


Худшие 10 лент для H1:


,feed_id,B0,H1,delta
31,"(522870,)",1.000000,0.356207,-0.643793
139,"(2042258,)",1.000000,0.356207,-0.643793
99,"(1488476,)",1.000000,0.430677,-0.569323
197,"(3067540,)",1.000000,0.500000,-0.500000
118,"(1707122,)",1.000000,0.630930,-0.369070
158,"(2404632,)",1.000000,0.630930,-0.369070
176,"(2719619,)",0.630930,0.356207,-0.274723
82,"(1287073,)",0.630930,0.386853,-0.244077
35,"(602139,)",0.755016,0.531193,-0.223823
170,"(2607161,)",0.630930,0.500000,-0.130930


Лучшие 10 лент для H1:


,feed_id,B0,H1,delta
148,"(2262542,)",0.500000,1.000000,0.500000
75,"(1172075,)",0.630930,1.000000,0.369070
88,"(1378685,)",0.630930,1.000000,0.369070
192,"(2957724,)",0.630930,1.000000,0.369070
20,"(386355,)",0.483813,0.797723,0.313910
45,"(779245,)",0.244651,0.430677,0.186026
134,"(1922818,)",0.289065,0.430677,0.141612
25,"(441243,)",0.221065,0.356207,0.135142
130,"(1853407,)",0.224244,0.356207,0.131963
121,"(1747095,)",0.500000,0.630930,0.130930


Средний выигрыш: 0.04967123630770767
Средний проигрыш: -0.08515011873968595
Медианное изменение: 0.0


In [6]:

import polars as pl

feed_id = 522870

for name, df in [("B0", pred_b0), ("H1", pred_h1)]:
    ranked = (
        df.filter(pl.col("feed_id") == feed_id)
          .sort("score", descending=True)
          .with_row_index("rank", offset=1)
    )

    print(f"\n{name} — положительные объявления:")

    columns = [
        "rank",
        "item_id",
        "category_id",
        "target",
        "score",
    ]

    if "hist_cat_count" in ranked.columns:
        columns.append("hist_cat_count")

    print(
        ranked.filter(pl.col("target") > 0)
              .select(columns)
    )



B0 — положительные объявления:
shape: (1, 5)
┌──────┬───────────┬─────────────┬────────┬──────────┐
│ rank ┆ item_id   ┆ category_id ┆ target ┆ score    │
│ ---  ┆ ---       ┆ ---         ┆ ---    ┆ ---      │
│ u32  ┆ i32       ┆ i8          ┆ i8     ┆ f64      │
╞══════╪═══════════╪═════════════╪════════╪══════════╡
│ 1    ┆ 101433613 ┆ 13          ┆ 1      ┆ 0.745604 │
└──────┴───────────┴─────────────┴────────┴──────────┘

H1 — положительные объявления:
shape: (1, 6)
┌──────┬───────────┬─────────────┬────────┬──────────┬────────────────┐
│ rank ┆ item_id   ┆ category_id ┆ target ┆ score    ┆ hist_cat_count │
│ ---  ┆ ---       ┆ ---         ┆ ---    ┆ ---      ┆ ---            │
│ u32  ┆ i32       ┆ i8          ┆ i8     ┆ f64      ┆ i64            │
╞══════╪═══════════╪═════════════╪════════╪══════════╪════════════════╡
│ 6    ┆ 101433613 ┆ 13          ┆ 1      ┆ 0.579075 ┆ 177            │
└──────┴───────────┴─────────────┴────────┴──────────┴────────────────┘
